<a href="https://colab.research.google.com/github/jp-signum/Binary-Classification-for-Climate-Press-Event-Deduplication/blob/main/binary_classification_for_climate_press_event_deduplication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Climate Press Event Deduplication

In [ ]:
!pip install lightning
!pip install openpyxl

  Using cached lightning-2.5.1.post0-py3-none-any.whl.metadata (39 kB)
Using cached lightning-2.5.1.post0-py3-none-any.whl (819 kB)


In [ ]:
# imports
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import lightning as pl
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
import torch.nn.functional as F
import numpy as np
import itertools
import random

# Initial Model Training

In [ ]:
# data
data = pd.read_json('training_data.json', orient="records", lines=True)

In [ ]:
# train / valid split
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, valid_idx = next(gss.split(data, groups=data["event_id"]))

train = data.iloc[train_idx]
valid = data.iloc[valid_idx]

In [ ]:
# pytorch model functions

class EventsDataset(Dataset):
    def __init__(self, data):
        self.labels = data['event_id']
        self.unique_labels = np.unique(self.labels)
        self.embeddings = data['content_embedding_raw']

    def __len__(self):
        return len(self.unique_labels)

    def __getitem__(self, idx):
        #item includes all article embeddings belonging to an event, and the event label
        event_label = self.unique_labels[idx]
        article_idx = self.labels[self.labels.isin([event_label])].index
        embeddings = [(torch.tensor(np.array(self.embeddings.loc[idx]), dtype=torch.float32), event_label) for idx in article_idx]
        return embeddings


def collate_fn(batch):
    if isinstance(batch[0], list):
        batch = list(itertools.chain(*batch))
    embeddings, labels = zip(*batch)
    embeddings = torch.stack(embeddings)
    labels = torch.tensor(labels)

    positive_pairs = []
    negative_pairs = []

    # Loop over embeddings in the batch
    for i in range(len(embeddings)):
        for j in range(i + 1, len(embeddings)):
            if labels[i] == labels[j]:
                positive_pairs.append((embeddings[i], embeddings[j]))
            else:
                negative_pairs.append((embeddings[i], embeddings[j]))

    # sample an equal number of negative pairs as positive pairs to balance
    negative_pairs = random.sample(negative_pairs, len(positive_pairs))

    all_pairs = positive_pairs + negative_pairs
    pair_labels = [1 for _ in positive_pairs] + [0 for _ in negative_pairs]

    combined = list(zip(all_pairs, pair_labels))
    np.random.shuffle(combined)

    pairs, pair_labels = zip(*combined)

    return torch.stack([pair[0] for pair in pairs]), torch.stack([pair[1] for pair in pairs]), torch.tensor(pair_labels)


def cosine(x, y):
    x_normalized = x / torch.norm(x, 2, dim=1, keepdim=True)
    y_normalized = y / torch.norm(y, 2, dim=1, keepdim=True)
    return torch.sum(x_normalized * y_normalized, dim=1)


class SimilarityModel(pl.LightningModule):
    def __init__(self, input_dim=3072, output_dim=256, l1_reg_weight=0.0001):
        super(SimilarityModel, self).__init__()
        self.encoder = nn.Linear(input_dim, output_dim)
        self.l1_reg_weight = l1_reg_weight

    def forward(self, x1, x2):
        embedding1 = self.encoder(x1)
        embedding2 = self.encoder(x2)
        return cosine(embedding1, embedding2)

    def training_step(self, batch, batch_idx):
        embedding1, embedding2, labels = batch

        # Forward pass to get cosine similarity
        cos_sim = self(embedding1, embedding2)

        # mean squared error with regularization
        loss = F.mse_loss(cos_sim, labels.float())
        l1_regularization = self.l1_reg_weight * torch.sum(torch.abs(self.encoder.weight))

        total_loss = loss + l1_regularization
        self.log("train_loss", total_loss, prog_bar=True, on_epoch=True)
        return total_loss

    def validation_step(self, batch, batch_idx):
        embedding1, embedding2, labels = batch

        # Forward pass to get cosine similarity
        cos_sim = self(embedding1, embedding2)
        total_loss = F.mse_loss(cos_sim, labels.float())
        self.log("val_loss", total_loss, prog_bar=True, on_epoch=True, logger=True)
        return total_loss

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.encoder.parameters(), lr=1e-3)
        return optimizer

In [ ]:
# create and train model

batch_size = 100

train_dataset = EventsDataset(data=train)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, collate_fn=collate_fn)

valid_dataset = EventsDataset(data=valid)
valid_dataloader = DataLoader(valid_dataset, batch_size=batch_size, collate_fn=collate_fn)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=0.00, patience=3, verbose=False, mode="min")

model = SimilarityModel()
tb_logger = pl.pytorch.loggers.TensorBoardLogger(save_dir="tensor_logs/")
trainer = pl.Trainer(callbacks=[early_stop_callback], logger=tb_logger, log_every_n_steps=1)
trainer.fit(model, train_dataloader, valid_dataloader)

INFO: Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO:lightning.pytorch.utilities.rank_zero:Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.11/dist-packages/lightning/pytorch/loops/utilities.py:73: `max_epochs` was not set. Setting it to 1000 epochs. To train without an epoch limit, set `max_epochs=-1`.
INFO: 
  | Name    | Type   | Params | Mode 
---------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
# save model weights

weight_array = model.encoder.weight.detach().numpy()
np.save('weights.npy', weight_array)

# New data

In [ ]:
# new data
new_data = pd.read_json('new_data.json', orient="records", lines=True)
analyst_review = pd.read_excel('analyst_merged_events.xlsx')
weight_array = np.load("weights.npy")

In [ ]:
new_data['content_embedding_transformed'] = new_data['content_embedding_raw'].apply(lambda x: np.dot(x, weight_array.T))

In [ ]:
# example of functions used to group similar articles into events

def get_similar_articles(
    data: pd.DataFrame, article_id: int, cut_off: float = 0.9
) -> pd.DataFrame:
    """Return a dataframe of all articles similar to input article"""
    df = data.copy()
    row = df.loc[df['article_id']==article_id]
    if row.empty:
        raise ValueError(f"Article ID {article_id} not found.")
    content_embedding_transformed = row['content_embedding_transformed'].values[0]
    content_embedding_transformed_array = content_embedding_transformed.reshape(1,-1)
    df['similarity'] = df['content_embedding_transformed'].apply(
        lambda x: cosine_similarity(np.array(x).reshape(1,-1), content_embedding_transformed_array)[0][0]
    )
    similar_articles = df[df['similarity']>cut_off].copy()

    return similar_articles


def get_or_create_event(df: pd.DataFrame, article_id: int, new_event_id: int) -> int:
    """Label events based on similar article dataframes"""
    similar_df = get_similar_articles(df, article_id)

    if similar_df.empty:
        # return a new_event_id
        return new_event_id

    similar_df = similar_df.dropna(subset=["event_id"])
    similar_article_ids = set(similar_df["article_id"])

    # potential for multiple options of events to tag the new article to
    event_options = {}

    for event_id in similar_df["event_id"].unique():
        event_articles = df[df["event_id"] == event_id]["article_id"].tolist()

        # Check if all articles in this event are part of the similar articles
        if all(article_id in similar_article_ids for article_id in event_articles):
            avg_similarity = similar_df[similar_df["event_id"] == event_id]["similarity"].mean()
            event_options[event_id] = avg_similarity

    # if multiple event options return the event_id with the maximum average similarity
    if event_options:
        return max(event_options, key=event_options.get)

    # no suitable existing event, return new event ID
    return new_event_id

In [ ]:
## a theoretical example of how we'd run get_or_create event on new articles
## in reality we'd be comparing the new data coming in to all recent articles and their events & auto-generating a new event id

new_event_id = 1
new_data['event_id'] = np.nan

for idx, row in new_data.iterrows():
    if pd.isna(row["event_id"]):
        assigned_event_id = get_or_create_event(new_data, row["article_id"], new_event_id)
        if assigned_event_id == new_event_id:
            new_event_id += 1
        new_data.loc[new_data["article_id"] == row["article_id"], "event_id"] = assigned_event_id

## Generate Positive Training Pairs from Analyst Feedback


This section extracts article ID pairs from `analyst_merged_events.xlsx` and uses the transformed embeddings from `new_data.json` to prepare labeled positive training data for retraining.

In [ ]:
import numpy as np

# build a lookup table for transformed embeddings
embedding_lookup = {
    row["article_id"]: row["content_embedding_transformed"]
    for _, row in new_data.iterrows()
}

# build positive pairs from analyst feedback
positive_pairs = []
for _, row in analyst_review.iterrows():
    try:
        merged_ids = [int(x.strip()) for x in str(row['Additional Merges Performed']).split(',')]
        base_id = int(row['New Event ID'])
        all_ids = [base_id] + merged_ids

        for i in range(len(all_ids)):
            for j in range(i + 1, len(all_ids)):
                id1, id2 = all_ids[i], all_ids[j]
                if id1 in embedding_lookup and id2 in embedding_lookup:
                    positive_pairs.append({
                      "embedding_1": embedding_lookup[id1],
                      "embedding_2": embedding_lookup[id2],
                      "article_id_1": id1,
                      "article_id_2": id2,
                      "label": 1
                  })
    except Exception as e:
        print(f"Error processing row: {e}")

print(f"Total positive pairs created: {len(positive_pairs)}")


Total positive pairs created: 7


## Preview Merged Event Groups from Analyst Feedback

In [ ]:
from collections import defaultdict

# recreate event → article ID groups from the Excel sheet
event_groups = defaultdict(list)

for _, row in analyst_review.iterrows():
    try:
        base_id = int(row['New Event ID'])
        merged_ids = [int(x.strip()) for x in str(row['Additional Merges Performed']).split(',')]
        full_group = [base_id] + merged_ids
        event_groups[base_id] = sorted(set(full_group))
    except:
        continue

# preview first 5 event groups
for i, (event_id, articles) in enumerate(event_groups.items()):
    print(f"Event {event_id}:")
    for aid in articles:
        print(f"  - {aid}")
    print("---")
    if i >= 4:
        break


Event 201:
  - 1
  - 3
  - 201
---
Event 204:
  - 4
  - 5
  - 204
---
Event 211:
  - 11
  - 12
  - 211
---
Event 215:
  - 15
  - 16
  - 215
---
Event 221:
  - 21
  - 22
  - 221
---


In [ ]:
# create set of all positive article ID pairs
positive_pair_keys = {
    tuple(sorted((pair["article_id_1"], pair["article_id_2"])))
    for pair in positive_pairs
}
print(f"Unique positive pair keys: {len(positive_pair_keys)}")


Unique positive pair keys: 7


## Generate Negative Training Pairs

This section randomly samples article ID pairs from `new_data.json` that are not part of any analyst-verified event group. These pairs are labeled as `0` for model training.


In [ ]:
import random

# build a set of all positive pair keys to avoid overlap
positive_pair_keys = {
    tuple(sorted((pair["article_id_1"], pair["article_id_2"])))
    for pair in positive_pairs
}

# build list of all usable article IDs
all_article_ids = list(embedding_lookup.keys())

# generate negative pairs until we match the number of positives
negative_pairs = set()
attempts = 0
max_attempts = len(positive_pairs) * 10

while len(negative_pairs) < len(positive_pairs) and attempts < max_attempts:
    id1, id2 = random.sample(all_article_ids, 2)
    if id1 == id2:
        continue
    key = tuple(sorted((id1, id2)))
    if key in positive_pair_keys or key in negative_pairs:
        continue
    if id1 in embedding_lookup and id2 in embedding_lookup:
        negative_pairs.add(key)
    attempts += 1

# format for training
negative_pairs_formatted = [
    {
        "embedding_1": embedding_lookup[id1],
        "embedding_2": embedding_lookup[id2],
        "article_id_1": id1,
        "article_id_2": id2,
        "label": 0
    }
    for id1, id2 in negative_pairs
]

print(f"Negative pairs created: {len(negative_pairs_formatted)}")


Negative pairs created: 7


## Combine Positive and Negative Training Pairs

This section merges the manually generated positive and negative training examples into one dataset for model retraining.


In [ ]:
# preview first 5 negative pairs with article IDs
for i, pair in enumerate(negative_pairs_formatted[:5]):
    print(f"Negative Pair {i+1}:")
    print(f"  Article ID 1: {pair['article_id_1']}")
    print(f"  Article ID 2: {pair['article_id_2']}")
    print(f"  Label: {pair['label']}")
    print("---")


Negative Pair 1:
  Article ID 1: 135
  Article ID 2: 177
  Label: 0
---
Negative Pair 2:
  Article ID 1: 34
  Article ID 2: 118
  Label: 0
---
Negative Pair 3:
  Article ID 1: 13
  Article ID 2: 67
  Label: 0
---
Negative Pair 4:
  Article ID 1: 92
  Article ID 2: 220
  Label: 0
---
Negative Pair 5:
  Article ID 1: 13
  Article ID 2: 15
  Label: 0
---


## Validate Negative Pairs Against Analyst Ground Truth

This section ensures that none of the generated negative training pairs accidentally include article ID combinations that were marked as matching events by analysts. This prevents false negatives from corrupting the training signal.


In [ ]:
# build all true positive article ID pairs from analyst feedback
true_positive_pairs = set()
for _, row in analyst_review.iterrows():
    try:
        merged_ids = [int(x.strip()) for x in str(row['Additional Merges Performed']).split(',')]
        base_id = int(row['New Event ID'])
        all_ids = [base_id] + merged_ids
        for i in range(len(all_ids)):
            for j in range(i + 1, len(all_ids)):
                true_positive_pairs.add(tuple(sorted((all_ids[i], all_ids[j]))))
    except:
        continue

# check negative pairs for accidental overlap
false_negatives = []
for pair in negative_pairs_formatted:
    aid1, aid2 = pair["article_id_1"], pair["article_id_2"]
    if tuple(sorted((aid1, aid2))) in true_positive_pairs:
        false_negatives.append((aid1, aid2))

if false_negatives:
    print(f"Found {len(false_negatives)} false negatives:")
    for fn in false_negatives:
        print(fn)
else:
    print("No false negatives detected.")


No false negatives detected.


## Combine Labeled Training Data

This section merges the cleaned positive and negative article pairs into a single dataset that will be used for retraining the model.


In [ ]:
# combine all labeled training pairs
all_pairs = positive_pairs + negative_pairs_formatted

print(f"Total training pairs: {len(all_pairs)}")
print(f"  Positive: {len(positive_pairs)}")
print(f"  Negative: {len(negative_pairs_formatted)}")

Total training pairs: 14
  Positive: 7
  Negative: 7


## Format Training Data for PyTorch

This section converts the combined labeled dataset into a PyTorch-compatible `Dataset` and `DataLoader` for model retraining.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

# convert all data into tensors
class PairDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        emb1 = torch.tensor(pair["embedding_1"], dtype=torch.float32)
        emb2 = torch.tensor(pair["embedding_2"], dtype=torch.float32)
        label = torch.tensor(pair["label"], dtype=torch.long)
        return emb1, emb2, label

# Initialize the dataset
train_dataset = PairDataset(all_pairs)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

print(f"Total batches: {len(train_loader)}")


Total batches: 4


## Retrain Model on Updated Labeled Dataset

This section retrains the model from scratch using the new labeled training data, based on analyst-verified positive examples and generated negatives.


In [ ]:
import lightning as L
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# Re-define the model (same architecture as before)
class PairClassifier(L.LightningModule):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Linear(input_dim * 2, 2)  # binary classifier

    def forward(self, x1, x2):
        x = torch.cat((x1, x2), dim=1)
        return self.encoder(x)

    def training_step(self, batch, batch_idx):
        x1, x2, y = batch
        logits = self(x1, x2)
        loss = F.cross_entropy(logits, y)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=1e-3)

# Detect embedding size
embedding_dim = len(all_pairs[0]["embedding_1"])

# Init model
model = PairClassifier(input_dim=embedding_dim)

# Train
trainer = L.Trainer(max_epochs=20, logger=False, enable_checkpointing=False)
trainer.fit(model, train_loader)


INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO: 
  | Name    | Type   | Params | Mode 
-------------------------------------------
0 | encoder | Linear | 1.0 K  | train
-------------------------------------------
1.0 K     Trainable params
0         Non-trainable params
1.0 K     Total params
0.004     Total estimated model params size (MB)
1         Modules in train mode
0         Modules in eval mode
INFO:lightning.pytorch.callbacks.model_summary:
  | Name    | Type   | Params | Mode 
-------------------------------------------
0 | encoder | Linear | 1.0 K  | train
-------------------------------------------
1.0 K     Trainable params
0         Non-trainable para

Training: |          | 0/? [00:00<?, ?it/s]

INFO: `Trainer.fit` stopped: `max_epochs=20` reached.
INFO:lightning.pytorch.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=20` reached.


## Evaluate Model on Training Data

This section runs the retrained model on the labeled training pairs and prints out predicted vs actual labels for inspection. This helps verify the model has learned the updated signal.


In [ ]:
model.eval()

correct = 0
results = []

with torch.no_grad():
    for x1, x2, y_true in train_loader:
        logits = model(x1, x2)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_true).sum().item()

        for i in range(len(y_true)):
            results.append({
                "true": y_true[i].item(),
                "pred": preds[i].item(),
                "prob_1": F.softmax(logits[i], dim=0)[1].item()
            })

accuracy = correct / len(train_dataset)
print(f"Training accuracy: {accuracy:.2f}")


Training accuracy: 0.86


In [ ]:
# show examples with prediction vs actual label
print("Predicted vs Actual (sorted by confidence):")
sorted_results = sorted(results, key=lambda r: abs(r["pred"] - r["true"]), reverse=True)

for i, row in enumerate(sorted_results):
    print(f"Example {i+1}: True={row['true']} | Pred={row['pred']} | Prob(class 1)={row['prob_1']:.2f}")


Predicted vs Actual (sorted by confidence):
Example 1: True=1 | Pred=0 | Prob(class 1)=0.50
Example 2: True=1 | Pred=0 | Prob(class 1)=0.50
Example 3: True=0 | Pred=0 | Prob(class 1)=0.49
Example 4: True=1 | Pred=1 | Prob(class 1)=0.50
Example 5: True=1 | Pred=1 | Prob(class 1)=0.51
Example 6: True=0 | Pred=0 | Prob(class 1)=0.49
Example 7: True=0 | Pred=0 | Prob(class 1)=0.48
Example 8: True=1 | Pred=1 | Prob(class 1)=0.51
Example 9: True=1 | Pred=1 | Prob(class 1)=0.50
Example 10: True=0 | Pred=0 | Prob(class 1)=0.49
Example 11: True=0 | Pred=0 | Prob(class 1)=0.47
Example 12: True=0 | Pred=0 | Prob(class 1)=0.49
Example 13: True=0 | Pred=0 | Prob(class 1)=0.50
Example 14: True=1 | Pred=1 | Prob(class 1)=0.51


## Retrain or otherwise update the model to benefit from the additional data

After retraining the model on the updated labeled dataset (7 analyst-verified positive pairs and 7 sampled negative pairs), I evaluated its predictions on the same dataset to verify learning.

**Training accuracy:** 0.86 (12 correct out of 14 predictions)

**False negatives:**  
The model predicted 2 positive pairs as negative. In both cases, the predicted class 1 probability was exactly 0.50, the decision threshold. That suggests the model wasn’t confident either way, and a small threshold change could flip the outcome. These were borderline, low-confidence cases, likely due to subtle semantic differences between the article pairs.

**No false positives:**  
All negative examples were correctly classified.

**Low confidence across predictions:**  
Most predicted probabilities for class 1 were between 0.47 and 0.51. That’s expected given the small dataset, the simple linear model, and how close many embeddings are in vector space.

**Interpretation:**  
The model adapted to the new signal. It shows high precision and low recall, meaning it avoids merging unless it’s sure. That behavior is safe but would need improvement in production if recall became a priority.

**Suggestions for improvement:**  
- Tune the decision threshold instead of always choosing the highest-scoring class (argmax)
- Add a non-linear projection layer or increase model depth  
- Try contrastive or triplet loss instead of binary classification  
- Collect more labeled examples, especially edge cases and hard negatives


## Comment on the training data, method, and evaluation metric used

The original method used cosine similarity and a static threshold to group articles. While simple, it didn’t leverage any supervision from labeled examples. I switched to training a binary classifier using analyst-verified article pairs. The model uses cross-entropy loss and predicts whether two articles refer to the same event.

Evaluation was done using accuracy, which is fine for a quick pass, but doesn't capture the business tradeoff between false merges and missed merges. In practice, precision and recall (or something like ROC-AUC) would give a better view of model performance, especially since false positives are more costly in this context.


## How you would implement a system of continuous learning

I’d build a weekly pipeline that collects new analyst merges, generates negative pairs, retrains the model using the updated dataset, and tracks performance on a held-out validation set. Model versions would only be promoted if accuracy (or another chosen metric) improves or stays stable.

Over time, I’d add uncertainty sampling, flagging low-confidence predictions for analyst review, and use those reviewed cases to improve the next training cycle. This would keep the system responsive to new data while reducing reliance on manual oversight.


## Limitations or challenges in using user feedback to improve the model

The main limitation is that analyst feedback only gives us positive pairs — article IDs that should be merged. We don’t get direct negatives, so we have to assume unmerged pairs are not matches. That assumption can introduce noise.

There’s also the issue of context. Analysts may merge based on things the model can’t see, like background knowledge, timing, or the source of the article. Without richer inputs or more examples, the model may miss that nuance and struggle on edge cases.


## Next Steps

If I had more time or more data, I'd expand the training set with more labeled pairs, especially borderline cases, and explore threshold tuning and ranking losses to better balance precision and recall. I’d also evaluate model generalization using new unseen event sets and experiment with richer embeddings.
